# Combat RL playground

This notebook is a hands-on tour of the **current** simulator and tensorizer
stack, aimed at the first steps of a combat RL agent. It is a playground, not a
training entry point.

The durable contract is symbolic:

1. `RunEnv` owns the authoritative run and exposes **one** legal-action list.
2. A policy / value network may see only `FairCombatObservation`.
3. `sts_sim.rl.tensor` turns that public decision into entity tokens and a
   dynamic action table. Tensors are an experiment representation, not a
   simulator persistence format.
4. `FairCombatPolicyValueNet` scores **the current legal rows**, not a giant
   global action head.
5. Training records stay symbolic JSON. Tensorization happens on load.

There is not yet a combat-root corpus, PUCT search, or Expert Iteration loop.
The "teacher" later in this notebook is a toy heuristic so you can feel the
record → tensor → train → rollout wiring.

Companion reading:

- `rl/docs/fair_combat_api_design.md`
- `rl/docs/fair_observation_hidden_state_audit.md`
- `simulator/docs/python_api.md` for the current non-RL API

This notebook's RL modules and optional Jupyter dependencies are not present
in the current base package. If an RL environment restores them, launch from
the repository root with the simulator Python project installed upstream:

```bash
uv run --project simulator/python jupyter lab rl/notebooks/combat_rl_playground.ipynb
```


## Setup

`sts_sim.rl` is an explicit extra import: the base package has no PyTorch
dependency. On this machine a 13-token Transformer is much faster with **one**
CPU thread than with the default thread pool, so pin that first.


In [ ]:
from __future__ import annotations

import torch

torch.set_num_threads(1)

from sts_sim import (
    DecisionUnavailableError,
    FairCombatObservation,
    RunEnv,
)
from sts_sim.notebook import action_label, show_decision
from sts_sim.rl import (
    SCALAR_INDEX,
    CombatModelConfig,
    CombatOutcome,
    FairCombatPolicyValueNet,
    RepositoryVersion,
    SymbolicCombatDataset,
    SymbolicTrainingRecord,
    VocabularyBuilder,
    collate_combat_tensors,
    collate_training_examples,
    fair_observation_digest,
    policy_value_loss,
    rollout_model_combat,
    summarize_rollouts,
    tensorize_combat,
)

print("torch", torch.__version__, "| CPU threads", torch.get_num_threads())


## 1. One environment, one action type

`RunEnv.combat_fixture()` is a tiny deterministic Ironclad fight against a
fixed 40 HP dummy. Use it for RL wiring. `RunEnv.new_ironclad(seed)` is a real
seeded run and is better as a later experiment.

`decision()` is atomic: fair observation, legal actions, and a revision. Always
submit an action from **that** decision (or an identical clone). Reconstructing
actions from slots or tuple positions is how stale-handle bugs happen.


In [ ]:
env = RunEnv.combat_fixture()
decision = env.decision()
assert isinstance(decision.observation, FairCombatObservation)

print(env)
print("revision", decision.revision, "| phase", decision.phase, "| kind", decision.kind)
print("n_actions", len(decision.actions))
print()
show_decision(decision)


## 2. Three state views

Fairness is a **projection**, not a second simulator.

| View | Use it for | Do not use it for |
|---|---|---|
| `observation()` / `decision().observation` | policy, tensorizer, training records | restoring a run |
| `full_state()` | debugging, omniscient research | model input or persistence |
| `snapshot()` | exact restore / search branching | model input |

Two hidden-distinct states can share one fair observation. A snapshot is not an
observation.


In [ ]:
fair = env.observation()
privileged = env.full_state()
snapshot = env.snapshot()
restored = RunEnv.from_snapshot(snapshot)

combat = privileged["combat"]
print("fair phase", fair.phase)
print("fair player HP", fair.player.hp, "/", fair.player.max_hp)
print("fair monster intent", fair.monsters[0].intent)
print()
print("privileged combat phase", combat["phase"])
print("privileged shuffle RNG counter", combat["shuffle_rng"]["counter"])
print("privileged monster RNG counter", combat["monster_rng"]["counter"])
print()
print("snapshot hash", snapshot.hash)
print("restored hash", restored.snapshot().hash)
print("fair observation has no RNG fields:", not hasattr(fair, "shuffle_rng"))


Clone before you step when you want to compare alternatives. The parent stays
put; the clone can consume the same bound action because it shares the
revision.


In [ ]:
parent = env.clone()
bash = next(
    action
    for action in decision.actions
    if action.kind == "play_hand_slot"
    and action.hand_slot is not None
    and next(
        card.card.content_key
        for card in decision.observation.hand
        if card.slot == action.hand_slot
    )
    == "Bash"
)
branched = env.clone()
after_bash = branched.step(bash).decision

print("parent still at revision", parent.decision().revision)
print("bash branch revision", after_bash.revision)
print("bash branch monster HP", after_bash.observation.monsters[0].hp)
print("parent monster HP", parent.decision().observation.monsters[0].hp)
print()
print("played:", action_label(decision, bash))


## 3. Tensorizer: entities in, current actions out

`tensorize_combat(observation, action_descriptors, vocabularies)` is the whole
model input. It does **not** import `RunEnv`, snapshots, or native handles.

Design choices you can poke at:

- Present entities become tokens: global, player, piles, cards, monsters,
  relics, potions, orbs, selection. There is no sparse vector over every card
  in the game.
- Hand / pile cards are unordered for model semantics. Public slots are
  references used by actions, not sequence positions.
- Powers and counters are pooled onto their owner token.
- The policy head scores only the legal rows in the order you passed them.
- Vocabularies are checkpoint-owned. Changing them changes input width.

`VocabularyBuilder` seeds the card catalogue up front, then records whatever
monsters, relics, powers, and action kinds it actually sees.


In [ ]:
def descriptors(decision):
    return tuple(action.descriptor() for action in decision.actions)


observation = decision.observation
actions = descriptors(decision)

builder = VocabularyBuilder()
builder.add(observation, actions)
vocab = builder.freeze()
encoded = tensorize_combat(observation, actions, vocab)

print("vocabulary fingerprint", vocab.fingerprint)
print("namespace sizes")
for name, frozen in vocab.namespaces.items():
    print(f"  {name:18} {len(frozen.tokens):4d}")
print()
print("entity tokens", tuple(encoded.entity_kind.shape), "| action rows", encoded.action_count)
print("OOV counts", dict(encoded.oov_counts) or "{}")


Entity 0 is always the unpadded global token. Action rows keep the original
legal-action order and point at entity indices through public slots. Hand cards
are stored in a canonical content order, so `action_source` will not match the
on-screen hand index.


In [ ]:
kind_vocab = vocab.namespaces["entity_kind"]
zone_vocab = vocab.namespaces["zone"]
content_namespace = {
    "global": "phase",
    "player": "entity_kind",
    "pile": "zone",
    "card": "card",
    "monster": "monster",
    "relic": "relic",
    "potion": "potion",
    "orb": "orb",
    "selection": "selection",
}

print(f"{'i':>3} {'kind':<10} {'zone':<10} {'parent':>6} content")
print("-" * 56)
for index, kind_id in enumerate(encoded.entity_kind.tolist()):
    kind = kind_vocab.tokens[kind_id]
    zone = zone_vocab.tokens[encoded.entity_zone[index].item()]
    parent = encoded.entity_parent[index].item()
    namespace = content_namespace[kind]
    content = vocab.namespaces[namespace].tokens[encoded.entity_content[index].item()]
    print(f"{index:3d} {kind:<10} {zone:<10} {parent:6d} {content}")

print()
print("action rows (model input order = decision.actions order)")
print(f"{'i':>3} {'legal action':<42} src tgt")
print("-" * 62)
for index, action in enumerate(decision.actions):
    src = encoded.action_source[index].item()
    tgt = encoded.action_target[index].item()
    src_s = "-" if not bool(encoded.action_source_mask[index]) else str(src)
    tgt_s = "-" if not bool(encoded.action_target_mask[index]) else str(tgt)
    print(f"{index:3d} {action_label(decision, action):<42} {src_s:>3} {tgt_s:>3}")

hp_col = SCALAR_INDEX["hp"]
print()
print("player token HP scalar", float(encoded.entity_scalars[1, hp_col]))
print("player token HP present", bool(encoded.entity_scalar_mask[1, hp_col]))


## 4. Tiny policy / value network

The network is a small Transformer over entity tokens. The `[global]` output
feeds the tanh value head. Each legal action is scored from

```text
[state, action_family, action_kind, source_entity or learned-empty, target_entity or learned-empty]
```

Padding rows (after collation) are masked to `-inf` before softmax. This
playground uses a shrunk config so cells stay interactive; the architecture
doc's ~318k parameter figure is the default `width=96, layers=2` model.


In [ ]:
config = CombatModelConfig(width=32, heads=4, layers=1, feedforward_width=64)
torch.manual_seed(0)
model = FairCombatPolicyValueNet(vocab, config).eval()
batch = collate_combat_tensors((encoded,))
output = model(batch)
probs = torch.softmax(output.logits, dim=-1)[0]

print("parameters", sum(parameter.numel() for parameter in model.parameters()))
print("logits", output.logits[0].tolist())
print("value", output.value[0].detach().item())
print()
print(f"{'action':<42} {'prob':>8}")
for action, probability in zip(decision.actions, probs.tolist(), strict=True):
    print(f"{action_label(decision, action):<42} {probability:8.3f}")


Permuting the legal-action list must permute the policy and leave the value
unchanged. The same is true of unordered entity collections after their action
slot references are remapped. That is a real test in `test_combat_model.py`;
here we just watch the action-order case.


In [ ]:
reversed_actions = actions[::-1]
reversed_batch = collate_combat_tensors((tensorize_combat(observation, reversed_actions, vocab),))
reversed_output = model(reversed_batch)

print("original logits ", [round(x, 4) for x in output.logits[0].tolist()])
print("reversed logits ", [round(x, 4) for x in reversed_output.logits[0].tolist()])
print("flipped original", [round(x, 4) for x in output.logits.flip(1)[0].tolist()])
print("values", output.value[0].detach().item(), reversed_output.value[0].detach().item())
print(
    "policy equivariant",
    torch.allclose(output.logits.flip(1), reversed_output.logits, atol=1e-6),
)
print("value invariant", torch.allclose(output.value, reversed_output.value, atol=1e-6))


## 5. Symbolic records, then tensors

Durable learning artifacts are `SymbolicTrainingRecord` JSON objects:
fair observation, public action descriptors, teacher visit counts, a named
scalar value target, the full outcome vector, planner identity, and a
repository attestation.

This cell uses a **toy** greedy teacher (`Bash` > `Strike` > `Defend` >
everything else). It is not the production beam planner and must not be
confused with search visit counts. It exists so the rest of the pipeline is
runnable without a root corpus.


In [ ]:
def heuristic_index(current, preferences: tuple[tuple[str, int], ...]) -> int:
    ranked: list[tuple[int, int, int]] = []
    for index, action in enumerate(current.actions):
        label = action_label(current, action)
        score = next((weight for name, weight in preferences if name in label), 0)
        ranked.append((score, -index, index))
    ranked.sort(reverse=True)
    return ranked[0][2]


def terminal_proxy(status: str, hp: int, max_hp: int) -> float:
    if status == "lost":
        return -1.0
    return max(-1.0, min(1.0, 0.5 + 0.5 * (hp / max_hp)))


def collect_heuristic_episode(
    preferences: tuple[tuple[str, int], ...],
    *,
    root_id: str,
    planner_name: str,
    max_steps: int = 40,
) -> tuple[list[SymbolicTrainingRecord], str, int]:
    run = RunEnv.combat_fixture()
    start = run.decision()
    assert isinstance(start.observation, FairCombatObservation)
    start_hp = start.observation.player.hp
    max_hp = start.observation.player.max_hp
    collected: list[tuple[FairCombatObservation, tuple, int, tuple[int, ...]]] = []
    status = "truncated"
    hp = start_hp
    repository = RepositoryVersion("notebook-local", False, "d" * 64)

    for _ in range(max_steps):
        current = run.decision()
        if not isinstance(current.observation, FairCombatObservation):
            status = "won"
            context = current.observation.context
            hp = context.player_hp or hp
            max_hp = context.player_max_hp or max_hp
            break
        if current.observation.phase in ("won", "lost"):
            status = current.observation.phase
            hp = current.observation.player.hp
            max_hp = current.observation.player.max_hp
            break

        public_actions = descriptors(current)
        chosen = heuristic_index(current, preferences)
        visits = tuple(8 if index == chosen else 1 for index in range(len(public_actions)))
        collected.append((current.observation, public_actions, chosen, visits))
        try:
            result = run.step(current.actions[chosen])
        except DecisionUnavailableError:
            # Losing this fixture currently fails closed when enumerating the
            # next public decision, even though observation() can already show
            # phase=lost. Treat that boundary as a lost episode.
            lost = run.observation()
            assert isinstance(lost, FairCombatObservation)
            status = "lost"
            hp = lost.player.hp
            max_hp = lost.player.max_hp
            break
        if result.terminal:
            nxt = result.decision
            if isinstance(nxt.observation, FairCombatObservation):
                hp = nxt.observation.player.hp
                max_hp = nxt.observation.player.max_hp
                status = nxt.observation.phase if nxt.observation.phase in ("won", "lost") else "won"
            else:
                status = "won"
                context = nxt.observation.context
                hp = context.player_hp or hp
                max_hp = context.player_max_hp or max_hp
            break

    outcome_status = status if status in {"won", "lost", "escaped", "truncated"} else "won"
    terminal_hp = 0 if outcome_status == "lost" else max(1, hp)
    outcome = CombatOutcome(
        status=outcome_status,
        terminal_hp=terminal_hp,
        terminal_max_hp=max_hp,
        hp_change=terminal_hp - start_hp,
        max_hp_change=0,
        gold_change=0,
        potion_slots=(None, None, None),
        counter_changes=(),
        terminal=outcome_status != "truncated",
        truncated=outcome_status == "truncated",
    )
    value = terminal_proxy(outcome.status, outcome.terminal_hp, outcome.terminal_max_hp)
    records = [
        SymbolicTrainingRecord(
            observation=obs,
            actions=public_actions,
            chosen_action_index=chosen,
            chosen_action=public_actions[chosen],
            teacher_visit_counts=visits,
            target_value=value,
            value_target_name="notebook_hp_proxy_v0",
            outcome=outcome,
            planner_name=planner_name,
            planner_version="notebook-1",
            search_config={"kind": "greedy_label_heuristic"},
            root_id=root_id,
            split_group_id="combat-fixture",
            teacher_pair_id=None,
            repository=repository,
            observation_digest=fair_observation_digest(obs),
        )
        for obs, public_actions, chosen, visits in collected
    ]
    return records, outcome.status, outcome.terminal_hp


BASH_FIRST = (("Bash", 3), ("Strike", 2), ("Defend", 1))
records, status, hp = collect_heuristic_episode(
    BASH_FIRST, root_id="fixture-bash-first", planner_name="heuristic_bash_first"
)
print(f"teacher finished: {len(records)} decisions, status={status}, hp={hp}")
print("first choices:")
for record in records:
    print(" ", record.chosen_action.kind, "index", record.chosen_action_index)

train_vocab_builder = VocabularyBuilder()
for record in records:
    train_vocab_builder.add(record.observation, record.actions)
train_vocab = train_vocab_builder.freeze()
print("train vocab fingerprint", train_vocab.fingerprint)


## 6. One behaviour-cloning step, then a few more

`SymbolicCombatDataset` tensorizes on `__getitem__`. `policy_value_loss` is
cross-entropy against the normalized visit distribution plus MSE on the named
scalar value. This is the loss the later Expert Iteration loop is supposed to
use; here the "search policy" is just a peaked heuristic.


In [ ]:
dataset = SymbolicCombatDataset(records, train_vocab)
batch = collate_training_examples([dataset[index] for index in range(len(dataset))])
print("batch entities", tuple(batch.decision.entity_kind.shape))
print("batch actions ", tuple(batch.decision.action_mask.shape))
print("value target  ", [round(value, 3) for value in batch.value_target.tolist()])

opening_env = RunEnv.combat_fixture()
opening_decision = opening_env.decision()
opening = tensorize_combat(
    opening_decision.observation,
    descriptors(opening_decision),
    train_vocab,
)
opening_batch = collate_combat_tensors((opening,))


def opening_probs(net: FairCombatPolicyValueNet) -> list[float]:
    was_training = net.training
    net.eval()
    with torch.no_grad():
        probabilities = torch.softmax(net(opening_batch).logits[0], dim=-1).tolist()
    net.train(was_training)
    return probabilities


torch.manual_seed(0)
learner = FairCombatPolicyValueNet(train_vocab, config)
optimizer = torch.optim.Adam(learner.parameters(), lr=1e-3)
before = opening_probs(learner)

losses: list[float] = []
learner.train()
for _ in range(40):
    output = learner(batch.decision)
    loss = policy_value_loss(
        output,
        batch.policy_target,
        batch.value_target,
        batch.decision.action_mask,
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))

after = opening_probs(learner)
print(f"loss {losses[0]:.4f} -> {losses[-1]:.4f}")
print()
print(f"{'opening action':<42} {'before':>8} {'after':>8}")
for action, p0, p1 in zip(opening_decision.actions, before, after, strict=True):
    print(f"{action_label(opening_decision, action):<42} {p0:8.3f} {p1:8.3f}")


## 7. Roll out the untrained vs trained net

`rollout_model_combat` samples a legal row from the network, then applies the
matching `Action` from `decision.actions`. Rust still owns legality and the
transition. The fixture is easy — even a random net usually wins — so watch
step counts and the opening policy more than win rate.


In [ ]:
learner.eval()
torch.manual_seed(1)
untrained = FairCombatPolicyValueNet(train_vocab, config).eval()

fresh_runs = [
    rollout_model_combat(
        RunEnv.combat_fixture(), untrained, train_vocab, generator_seed=seed, max_steps=40
    )
    for seed in range(8)
]
trained_runs = [
    rollout_model_combat(
        RunEnv.combat_fixture(), learner, train_vocab, generator_seed=seed, max_steps=40
    )
    for seed in range(8)
]

print("untrained", summarize_rollouts(fresh_runs))
print("trained  ", summarize_rollouts(trained_runs))
print("untrained steps", [run.steps for run in fresh_runs])
print("trained steps  ", [run.steps for run in trained_runs])


## 8. Current limits you will hit if you push past the fixture

These are real, documented gaps, not notebook bugs:

- **No Expert Iteration yet.** Nothing outside tests constructs a production
  `SymbolicTrainingRecord`. There is no PUCT, replay buffer, or evaluation
  harness.
- **The fair boundary cannot finish every combat.** A lost fixture (always
  `End turn`) dies at `DecisionUnavailableError` while `observation()` already
  shows `phase=lost`. Seeded Ironclad runs often die the same way on the
  post-combat `Proceed`. Combat episodes currently need to stop in Python
  before requesting that next decision.
- **Two teachers exist, neither is wired here.** The live planner and the
  Python beam are not the heuristic above.
- **Privileged search, fair network.** The intended bootstrap still plans on
  one true hidden state. The net must not see `full_state()` or snapshots.

The next cell reproduces the lost-combat boundary so you can see both views.


In [ ]:
doomed = RunEnv.combat_fixture()
try:
    for step_index in range(20):
        current = doomed.decision()
        print(f"{step_index:02} hp={current.observation.player.hp} -> End turn")
        end_turn = next(action for action in current.actions if action.kind == "end_turn")
        doomed.step(end_turn)
except DecisionUnavailableError as error:
    lost = doomed.observation()
    print("legal_actions/decision failed:", error)
    print("observation still works:", lost.phase, "HP", lost.player.hp)
    print("privileged combat phase:", doomed.full_state()["combat"]["phase"])


## 9. Playground

Step the fixture yourself, or replace `choose` with any function of the fair
observation and legal actions. Keep the returned `Action` object intact.

Ideas:

- Bias the heuristic toward `Defend` and retrain; terminal HP goes *up* on this
  dummy because it always hits for 6.
- Clone every legal action at the root and print monster HP / player HP after
  one step (a 1-ply privileged lookahead — fine for study, not a fair agent).
- Add a potion with `env.add_potion` and watch a new potion token plus
  `use_potion_slot` / `discard_potion_slot` rows appear.
- Call `builder.add(...)` on a second observation and `freeze()` again: the
  fingerprint changes, and a model built on the old vocab will reject the new
  batch.


In [ ]:
def score_decision(current):
    tensors = tensorize_combat(current.observation, descriptors(current), train_vocab)
    with torch.no_grad():
        logits = learner(collate_combat_tensors((tensors,))).logits[0]
    probabilities = torch.softmax(logits, dim=-1)
    for action, probability in zip(current.actions, probabilities.tolist(), strict=True):
        print(f"{probability:6.3f}  {action_label(current, action)}")
    return current.actions[int(probabilities.argmax())]


play = RunEnv.combat_fixture()
current = play.decision()
show_decision(current)
print()
picked = score_decision(current)
print()
print("stepping", action_label(current, picked))
show_decision(play.step(picked).decision)


In [ ]:
root = RunEnv.combat_fixture()
root_decision = root.decision()
print(f"{'action':<42} {'player':>7} {'monster':>8} {'energy':>6}")
for action in root_decision.actions:
    child = root.clone()
    nxt = child.step(action).decision
    obs = nxt.observation
    if isinstance(obs, FairCombatObservation):
        monster_hp = obs.monsters[0].hp if obs.monsters else "-"
        print(
            f"{action_label(root_decision, action):<42} "
            f"{obs.player.hp:7d} {monster_hp:8} {obs.player.energy:6d}"
        )
    else:
        print(f"{action_label(root_decision, action):<42} left combat ({obs.kind})")


When you want a full seeded run instead of the dummy fight, start from
`RunEnv.new_ironclad("ABC123")` and the other playground notebook. Stay on
`decision().observation` for anything that looks like a policy input.
